# Cryptocurrency Wallet Risk Classification - Training from Pre-fetched Data

This notebook trains a Graph Neural Network (GNN) model using pre-fetched transaction data from:
- `bb_transactions.csv` - Benign Bitcoin transactions
- `cb_transactions.csv` - Criminal Bitcoin transactions

This avoids the slow API fetching process by using already collected transaction edges.

In [1]:
# IMPORTS
import pandas as pd
import numpy as np
import os
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION ---
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
REAL_CATS_DATA_DIR = os.path.join(PROJECT_ROOT, 'Real_Cats_data')
FULL_DATASET_DIR = os.path.join(REAL_CATS_DATA_DIR, 'full_dataset')

# Pre-fetched transaction files
PATH_BB_TRANSACTIONS = os.path.join(REAL_CATS_DATA_DIR, 'bb_transactions.csv')
PATH_CB_TRANSACTIONS = os.path.join(REAL_CATS_DATA_DIR, 'cb_transactions.csv')

# Wallet feature files
PATH_BENIGN = os.path.join(REAL_CATS_DATA_DIR, 'BB.tsv')
PATH_CRIMINAL = os.path.join(REAL_CATS_DATA_DIR, 'CB.tsv')

os.makedirs(FULL_DATASET_DIR, exist_ok=True)

# Features whitelist
NUMERIC_FEATURES = [
    'balance', 
    'total_received_USD', 
    'total_sent_USD',
    'lifetime',
    'transaction_number',
    'activity_w',
    'activity_d',
    'activity_time',
    'transaction_fee',
    'transaction_fee_Variance',
    'received_Variance_USD',
    'sent_Variance_USD',
    'total_input_slots',
    'total_output_slots',
    'payment_transactions',
    'receipt_transactions',
    'flow_ratio',
    'fan_ratio'
]

print("Configuration loaded.")
print(f"BB Transactions: {PATH_BB_TRANSACTIONS}")
print(f"CB Transactions: {PATH_CB_TRANSACTIONS}")

Configuration loaded.
BB Transactions: D:\Projects\final_project\Real_Cats_data\bb_transactions.csv
CB Transactions: D:\Projects\final_project\Real_Cats_data\cb_transactions.csv


In [2]:
def load_transaction_edges():
    """Load pre-fetched transaction edges from CSV files"""
    print("1. Loading Pre-fetched Transaction Edges...")
    
    # Load benign transactions
    df_bb = pd.read_csv(PATH_BB_TRANSACTIONS)
    df_bb['label'] = 0  # benign
    print(f"   Loaded {len(df_bb)} benign transactions")
    
    # Load criminal transactions
    df_cb = pd.read_csv(PATH_CB_TRANSACTIONS)
    df_cb['label'] = 1  # criminal
    print(f"   Loaded {len(df_cb)} criminal transactions")
    
    # Combine edges
    df_edges = pd.concat([df_bb, df_cb], ignore_index=True)
    print(f"   Total edges: {len(df_edges)}")
    
    return df_edges

df_edges = load_transaction_edges()
print("\nEdge columns:", df_edges.columns.tolist())
print(df_edges.head())

1. Loading Pre-fetched Transaction Edges...
   Loaded 65197 benign transactions


   Loaded 45793 criminal transactions
   Total edges: 110990

Edge columns: ['source', 'target', 'weight', 'timestamp', 'direction', 'txid', 'wallet_label', 'label']
                                       source  \
0          1N52wHoVR79PMDishab2XmRHsbekCdGquK   
1  bc1q42u72wmw62ftchmfrqfffa7arnhg4q0r2cvppx   
2  bc1q4hcppml73khank3wtdckgpglewytdw9e4dmy5r   
3  bc1q42u72wmw62ftchmfrqfffa7arnhg4q0r2cvppx   
4          3C82f98QWtdDchqyH7FfmdNoEKakAgDuSk   

                                       target  weight   timestamp direction  \
0          19E6qdJvejZDq5pGYoJ47MMjcbhkkKFpHo  369787  1700702679  incoming   
1          13zKyMrnQmWxFezGsE1q7wUxZsxZN2eWgt   33883  1708462340  outgoing   
2  bc1q42u72wmw62ftchmfrqfffa7arnhg4q0r2cvppx   36708  1708461271  incoming   
3  bc1q93whp2kdwwu06aguc40aeyksmq80zhp4478gd6  152243  1703612650  outgoing   
4  bc1q42u72wmw62ftchmfrqfffa7arnhg4q0r2cvppx  173380  1703606175  incoming   

                                                txid wallet_labe

In [3]:
def load_wallet_features():
    """Load wallet behavioral features from TSV files"""
    print("2. Loading Wallet Features...")
    
    # Load benign wallets
    df_b = pd.read_csv(PATH_BENIGN, sep='\t')
    df_b['label'] = 0
    print(f"   Loaded {len(df_b)} benign wallet records")
    
    # Load criminal wallets
    df_c = pd.read_csv(PATH_CRIMINAL, sep='\t')
    df_c['label'] = 1
    print(f"   Loaded {len(df_c)} criminal wallet records")
    
    return df_b, df_c

df_benign_wallets, df_criminal_wallets = load_wallet_features()

2. Loading Wallet Features...


   Loaded 90164 benign wallet records


   Loaded 90597 criminal wallet records


In [4]:
def extract_wallets_from_edges(df_edges):
    """Extract unique labeled wallets from the transaction edges"""
    print("3. Extracting Labeled Wallets from Edges...")
    
    # Get unique wallets that appear in edges with their labels
    # For outgoing transactions, the source is the labeled wallet
    # For incoming transactions, the target is the labeled wallet
    
    wallets_out = df_edges[df_edges['direction'] == 'outgoing'][['source', 'label']].rename(columns={'source': 'address'})
    wallets_in = df_edges[df_edges['direction'] == 'incoming'][['target', 'label']].rename(columns={'target': 'address'})
    
    # Combine and drop duplicates
    labeled_wallets = pd.concat([wallets_out, wallets_in]).drop_duplicates(subset='address')
    
    benign_addrs = set(labeled_wallets[labeled_wallets['label'] == 0]['address'])
    criminal_addrs = set(labeled_wallets[labeled_wallets['label'] == 1]['address'])
    
    print(f"   Unique benign wallets in edges: {len(benign_addrs)}")
    print(f"   Unique criminal wallets in edges: {len(criminal_addrs)}")
    
    return labeled_wallets, benign_addrs, criminal_addrs

labeled_wallets_df, benign_addrs, criminal_addrs = extract_wallets_from_edges(df_edges)

3. Extracting Labeled Wallets from Edges...
   Unique benign wallets in edges: 824
   Unique criminal wallets in edges: 1000


In [5]:
def merge_wallet_features(labeled_wallets_df, df_benign_wallets, df_criminal_wallets):
    """Merge wallet features with labeled wallets from edges"""
    print("4. Merging Wallet Features...")
    
    # Combine all wallet features
    common_cols = list(set(df_benign_wallets.columns) & set(df_criminal_wallets.columns))
    df_all_features = pd.concat([df_benign_wallets[common_cols], df_criminal_wallets[common_cols]], ignore_index=True)
    
    # Merge with labeled wallets from edges
    df_nodes = labeled_wallets_df.merge(df_all_features, on='address', how='left', suffixes=('', '_feat'))
    
    # Use label from edges if label_feat is different
    if 'label_feat' in df_nodes.columns:
        df_nodes['label'] = df_nodes['label'].fillna(df_nodes['label_feat'])
        df_nodes = df_nodes.drop(columns=['label_feat'])
    
    print(f"   Merged nodes: {len(df_nodes)}")
    print(f"   Nodes with features: {df_nodes.dropna(subset=['balance']).shape[0]}")
    
    return df_nodes

df_nodes = merge_wallet_features(labeled_wallets_df, df_benign_wallets, df_criminal_wallets)
print(f"\nNode columns: {df_nodes.columns.tolist()[:10]}...")

4. Merging Wallet Features...


   Merged nodes: 1824
   Nodes with features: 1824

Node columns: ['address', 'label', 'total_sent_BTC', 'gs_checked', 'transaction_fee', 'total_output_slots', 'total_input_slots', 'sent_Variance_BTC', 'total_sent_USD', 'total_received_BTC']...


In [6]:
def perform_feature_engineering(df):
    """Engineer additional features and clean data"""
    print("5. Engineering Features...")
    
    # Create Ratios
    df['flow_ratio'] = df['total_sent_USD'] / (df['total_received_USD'] + 1e-5)
    df['fan_ratio'] = df['total_output_slots'] / (df['total_input_slots'] + 1e-5)
    
    # Fill missing values with 0
    for col in NUMERIC_FEATURES:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        else:
            df[col] = 0.0
    
    print(f"   Engineered {len(NUMERIC_FEATURES)} features")
    return df

df_nodes = perform_feature_engineering(df_nodes)

5. Engineering Features...
   Engineered 18 features


In [7]:
def build_tensors(df_nodes, df_edges):
    """Build PyTorch Geometric tensors"""
    print("6. Building Tensors...")
    
    # Clean edges and add log weight
    df_edges = df_edges.drop_duplicates(subset=['source', 'target', 'timestamp'])
    w = pd.to_numeric(df_edges['weight'], errors='coerce').fillna(0)
    df_edges['weight_log'] = np.log1p(np.maximum(0, w))
    
    # Map addresses to indices
    known_addrs = df_nodes['address'].tolist()
    ghost_addrs = list(set(df_edges['source']).union(set(df_edges['target'])) - set(known_addrs))
    all_nodes = known_addrs + ghost_addrs
    addr_map = {addr: i for i, addr in enumerate(all_nodes)}
    
    print(f"   Total nodes: {len(all_nodes)} ({len(known_addrs)} known + {len(ghost_addrs)} ghosts)")
    
    # Scale features (log scale then standardize)
    df_scaled = df_nodes.copy()
    for c in NUMERIC_FEATURES:
        if c in df_scaled.columns:
            series = pd.to_numeric(df_scaled[c], errors='coerce').fillna(0)
            df_scaled[c] = np.log1p(np.maximum(0, series))
        else:
            df_scaled[c] = 0.0
    
    scaler = StandardScaler()
    known_feats = scaler.fit_transform(df_scaled[NUMERIC_FEATURES])
    
    # Build feature matrix
    x_np = np.zeros((len(all_nodes), len(NUMERIC_FEATURES)))
    x_np[0:len(known_addrs)] = known_feats
    x = torch.tensor(x_np, dtype=torch.float)
    
    # Build labels (-1 for ghost nodes)
    y_np = np.full(len(all_nodes), -1)
    y_np[0:len(known_addrs)] = df_nodes['label'].values.astype(int)
    y = torch.tensor(y_np, dtype=torch.long)
    
    # Build edge index
    src = df_edges['source'].map(addr_map).values
    dst = df_edges['target'].map(addr_map).values
    mask = (~np.isnan(src)) & (~np.isnan(dst))
    edge_index = torch.tensor(np.array([src[mask], dst[mask]]), dtype=torch.long)
    edge_attr = torch.tensor(df_edges['weight_log'].to_numpy()[mask].reshape(-1, 1), dtype=torch.float)
    
    print(f"   X shape: {x.shape}")
    print(f"   Edge index shape: {edge_index.shape}")
    print(f"   Labels: {(y == 0).sum()} benign, {(y == 1).sum()} criminal")
    
    # Save tensors
    torch.save(x, os.path.join(FULL_DATASET_DIR, 'x.pt'))
    torch.save(y, os.path.join(FULL_DATASET_DIR, 'y.pt'))
    torch.save(edge_index, os.path.join(FULL_DATASET_DIR, 'edge_index.pt'))
    torch.save(edge_attr, os.path.join(FULL_DATASET_DIR, 'edge_attr.pt'))
    
    print(f"   Tensors saved to {FULL_DATASET_DIR}")
    
    return x, y, edge_index, edge_attr

x, y, edge_index, edge_attr = build_tensors(df_nodes, df_edges)

6. Building Tensors...
   Total nodes: 92554 (1824 known + 90730 ghosts)


C:\Users\orimood\AppData\Local\Temp\ipykernel_6092\2495938783.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_edges['weight_log'] = np.log1p(np.maximum(0, w))


   X shape: torch.Size([92554, 18])
   Edge index shape: torch.Size([2, 104371])
   Labels: 824 benign, 1000 criminal
   Tensors saved to D:\Projects\final_project\Real_Cats_data\full_dataset


In [8]:
class CryptoGNN(torch.nn.Module):
    """Graph Attention Network for cryptocurrency wallet classification"""
    def __init__(self, num_node_features, num_edge_features, hidden_channels, num_classes):
        super().__init__()
        self.conv1 = GATv2Conv(num_node_features, hidden_channels, heads=2, edge_dim=num_edge_features)
        self.conv2 = GATv2Conv(hidden_channels * 2, hidden_channels, heads=1, edge_dim=num_edge_features)
        self.classifier = torch.nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, edge_attr):
        h = self.conv1(x, edge_index, edge_attr=edge_attr)
        h = h.relu()
        h = F.dropout(h, p=0.3, training=self.training)
        h = self.conv2(h, edge_index, edge_attr=edge_attr)
        h = h.relu()
        return self.classifier(h)

In [9]:
def train_model(x, y, edge_index, edge_attr, epochs=100):
    """Train the GNN model"""
    print("7. Training Model...")
    
    # Get valid (labeled) indices
    valid_idx = torch.where(y != -1)[0].numpy()
    print(f"   Valid labeled nodes: {len(valid_idx)}")
    
    # Train/test split
    train_idx, test_idx = train_test_split(valid_idx, test_size=0.2, stratify=y[valid_idx], random_state=42)
    print(f"   Train: {len(train_idx)}, Test: {len(test_idx)}")
    
    # Setup device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"   Device: {device}")
    
    # Initialize model
    model = CryptoGNN(x.shape[1], edge_attr.shape[1], hidden_channels=32, num_classes=2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    criterion = torch.nn.CrossEntropyLoss()
    
    # Move data to device
    x = x.to(device)
    y = y.to(device)
    edge_index = edge_index.to(device)
    edge_attr = edge_attr.to(device)
    
    # Training loop
    model.train()
    best_acc = 0
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        out = model(x, edge_index, edge_attr)
        loss = criterion(out[train_idx], y[train_idx])
        loss.backward()
        optimizer.step()
        
        # Evaluate periodically
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                pred = out.argmax(dim=1)
                train_acc = (pred[train_idx] == y[train_idx]).float().mean()
                test_acc = (pred[test_idx] == y[test_idx]).float().mean()
                
                if test_acc > best_acc:
                    best_acc = test_acc
                
                print(f"   Epoch {epoch:3d} | Loss: {loss.item():.4f} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")
            model.train()
    
    # Final evaluation
    model.eval()
    with torch.no_grad():
        out = model(x, edge_index, edge_attr)
        pred = out.argmax(dim=1)
        
        test_acc = (pred[test_idx] == y[test_idx]).float().mean()
        
        # Confusion matrix
        tp = ((pred[test_idx] == 1) & (y[test_idx] == 1)).sum()
        tn = ((pred[test_idx] == 0) & (y[test_idx] == 0)).sum()
        fp = ((pred[test_idx] == 1) & (y[test_idx] == 0)).sum()
        fn = ((pred[test_idx] == 0) & (y[test_idx] == 1)).sum()
        
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
        print(f"\n   === Final Results ===")
        print(f"   Test Accuracy: {test_acc:.4f}")
        print(f"   Precision: {precision:.4f}")
        print(f"   Recall: {recall:.4f}")
        print(f"   F1 Score: {f1:.4f}")
        print(f"   Confusion Matrix: TP={tp}, TN={tn}, FP={fp}, FN={fn}")
    
    # Save model
    model_path = os.path.join(PROJECT_ROOT, 'models', 'crypto_gnn_model.pt')
    torch.save(model, model_path)
    print(f"\n   Model saved to: {model_path}")
    
    return model

model = train_model(x, y, edge_index, edge_attr, epochs=100)

7. Training Model...
   Valid labeled nodes: 1824
   Train: 1459, Test: 365
   Device: cuda


   Epoch   0 | Loss: 0.7240 | Train Acc: 0.5168 | Test Acc: 0.5178


   Epoch  10 | Loss: 0.5097 | Train Acc: 0.7416 | Test Acc: 0.7342


   Epoch  20 | Loss: 0.4468 | Train Acc: 0.8060 | Test Acc: 0.8411


   Epoch  30 | Loss: 0.4108 | Train Acc: 0.8307 | Test Acc: 0.8603


...

   Epoch  60 | Loss: 0.3453 | Train Acc: 0.8609 | Test Acc: 0.8822


   Epoch  70 | Loss: 0.3339 | Train Acc: 0.8725 | Test Acc: 0.8849


   Epoch  80 | Loss: 0.3323 | Train Acc: 0.8663 | Test Acc: 0.8877


   Epoch  90 | Loss: 0.3236 | Train Acc: 0.8746 | Test Acc: 0.8849



   === Final Results ===
   Test Accuracy: 0.8986
   Precision: 0.8590
   Recall: 0.9750
   F1 Score: 0.9133
   Confusion Matrix: TP=195, TN=133, FP=32, FN=5

   Model saved to: D:\Projects\final_project\models\crypto_gnn_model.pt


In [10]:
print("="*60)
print("Training Complete!")
print("="*60)
print(f"\nModel saved to: {os.path.join(PROJECT_ROOT, 'models', 'crypto_gnn_model.pt')}")
print(f"Tensors saved to: {FULL_DATASET_DIR}")

Training Complete!

Model saved to: D:\Projects\final_project\models\crypto_gnn_model.pt
Tensors saved to: D:\Projects\final_project\Real_Cats_data\full_dataset
